# 02 — Deep Agents 解释器

**来源:** [LangChain Docs — Interpreters](https://docs.langchain.com/oss/python/deepagents/interpreters)

解释器（Interpreter）为 Agent 提供一个可编程的工作空间，让 Agent 编写代码来表达意图，在内存中执行并返回结果。

> **实验性功能。** API 和行为可能在不同版本间变化。需要 `langchain-quickjs>=0.1.0` 和 `Python >=3.11`。

**解释器 vs 沙箱：**
- **沙箱** — 作用于环境：运行命令、安装依赖、编辑文件
- **解释器** — 作用于 Agent 循环内部：编排工具、保持状态、决定哪些信息返回给模型

## 1. 何时使用解释器？

大多数 Agent 工作在模型推理和工具执行之间交替。但当 Agent 需要组合多个步骤、处理结构化数据或管理中间状态时，解释器更合适。

**典型场景：**
- 用代码组合多个工具调用（循环、分支、重试、并发）
- 从代码中编排子 Agent：拆分任务、存储结果、拼接合成
- 在运行时状态中保留中间值，避免每次临时结果都经过模型上下文
- 确定性结构化数据转换（排序、分组、解析、校验、评分、聚合）
- 探索大变量空间，只返回选中的证据/摘要/输出给模型

**架构流程：**

```mermaid
graph LR
    Model[Model] --> Program[Writes a small program]
    Program --> Runtime[Interpreter runtime]
    Runtime --> Tools[Calls tools and updates variables]
    Runtime --> Result[Yields compact result]
    Result --> Model
```

## 2. 执行路径选择

| 需求 | 推荐方式 |
|------|----------|
| 1-2 个简单外部调用 | 普通 Tool Calling |
| 需要循环、分支、重试或聚合的小程序 | 解释器 |
| 从代码中调用大量工具 | 解释器 + PTC |
| 跨线程的可复用 Helper | 解释器 + Interpreter Skills |
| Shell 命令、包安装、测试、OS 文件系统 | 沙箱 |

解释器基于 **QuickJS** — 一个轻量级 JavaScript 运行时，默认不暴露宿主机文件系统、网络、Shell、包管理或时钟 API。

## 3. 添加解释器到 Agent

In [ ]:
# 安装: pip install -U "deepagents[quickjs]"

from deepagents import create_deep_agent
from langchain_quickjs import CodeInterpreterMiddleware
from dotenv import load_dotenv
load_dotenv()

agent = create_deep_agent(
    model="deepseek-v4-flash",
    middleware=[CodeInterpreterMiddleware()],
)

print("Agent with interpreter middleware created.")


## 4. 在解释器中运行代码

中间件会添加一个 `eval` 工具。Agent 可以在持久化的上下文中运行 TypeScript 代码，`console.log` 会被捕获，返回最后一个表达式的值：

In [ ]:
// 解释器中运行的 TypeScript 代码示例
const rows = [
  { team: "alpha", score: 8 },
  { team: "beta", score: 13 },
  { team: "alpha", score: 21 },
];

const totals = rows.reduce((acc, row) => {
  acc[row.team] = (acc[row.team] ?? 0) + row.score;
  console.log(`${row.team} score: ${acc[row.team]}`)
  return acc;
}, {});

totals;  // 返回值会被捕获

> **状态持久化：** 解释器的状态在同一次线程中的不同轮次之间自动保存（快照），下次执行时恢复。

## 5. 程序化工具调用（PTC）

PTC（Programmatic Tool Calling）将选中的 Agent 工具暴露到解释器中，以 `tools` 命名空间下的异步 JavaScript 函数形式存在。

**工作原理：**
1. 你选择哪些工具可通过 PTC 调用（白名单）
2. 中间件将这些工具暴露为 `tools.xxx` 异步函数
3. Agent 编写代码调用 `await tools.xxx(...)`
4. 中间件执行工具桥接，返回结果
5. 模型只收到最终的解释器输出，而不是每个中间值

**工具名转换：** 蛇形命名（snake_case）转为驼峰命名（camelCase）。例如 `web_search` 变为 `tools.webSearch(...)`。

### 5.1 启用 PTC

In [ ]:
from deepagents import create_deep_agent
from langchain_quickjs import CodeInterpreterMiddleware

# 通过 ptc 参数指定白名单
agent = create_deep_agent(
    model="deepseek-v4-flash",
    middleware=[CodeInterpreterMiddleware(ptc=["task"])],
)

print("PTC enabled with 'task' tool allowlisted.")

### 5.2 PTC 使用模式

**批量处理：** 循环遍历多个输入，为每个输入调用一个工具

**并行处理：** 使用 `Promise.all` 同时调用独立工具

**条件逻辑：** 根据先前结果选择下一步调用

**提前终止：** 满足条件后停止调用

**数据过滤：** 只将相关的行/片段/错误/摘要返回给模型

**递归编排：** 反复调用 `task` 工具，然后在代码中组合子 Agent 结果

### 5.3 并行子 Agent 示例

In [ ]:
// 从解释器中并行启动多个子 Agent
const topics = ["retrieval", "memory", "evaluation"];

const reports = await Promise.all(
  topics.map((topic) =>
    tools.task({
      description: `Research ${topic} in Deep Agents and return three concise findings.`,
      subagent_type: "general-purpose",
    }),
  ),
);

reports.join("\n\n");

### 5.4 错误处理

In [ ]:
// 在代码中处理子 Agent 失败
try {
  const report = await tools.task({
    description: "Check the migration notes and return breaking changes.",
    subagent_type: "general-purpose",
  });
  console.log(report);
} catch (error) {
  console.log(`Subagent failed: ${error.message}`);
}

> **注意：** PTC 调用通过解释器桥接执行，不经过正常的工具调用路径。因此 `interrupt_on` 审批工作流对 PTC 调用的工具**不生效**。

## 6. 递归语言模型（Recursive LM）

递归语言模型使用解释器作为**分解工作空间**。模型将大型输入或工作集保存在运行时变量中，编写代码检查和拆分，调用子 Agent 处理小片段，然后在代码中拼接结果。

**核心分离：**
- **变量空间** — 解释器中存储的信息
- **Agent 上下文** — 模型实际处理的内容

```mermaid
flowchart TB
    Model[Model] --> Runtime[Interpreter variable space]
    subgraph Runtime[Interpreter variable space]
        Data[Long input and working notes]
        Select{Select next slice}
        Task[Call subagent]
        Store[Store subagent result]
        Stitch[Stitch results in code]
        Data --> Select
        Select --> Task
        Task --> Store
        Store --> Select
        Store --> Stitch
    end
    Stitch --> Answer[Compact synthesis]
    Answer --> Model
```

## 7. 快照与时间旅行

`CodeInterpreterMiddleware` 默认在每个 Agent 运行后对解释器状态做快照，下次运行前恢复。

**生命周期：**
1. 轮次开始 → 恢复该线程的最新解释器快照
2. Agent 调用 `eval` → 代码可以读取/修改解释器变量
3. Agent 运行结束 → 将更新后的状态快照保存到 Graph 状态
4. 下一轮 → 从恢复的状态开始，而非空运行时

**与 LangGraph Time Travel 配合：** 当 Graph 使用 Checkpointer 时，恢复 Graph 检查点可以还原存储在 Graph 状态中的解释器快照，从而实现调试和回放。

In [ ]:
from deepagents import create_deep_agent
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

agent = create_deep_agent(
    model="deepseek-v4-flash",
    checkpointer=checkpointer,
    middleware=[
        CodeInterpreterMiddleware(
            snapshot_between_turns=True,  # 默认开启
        )
    ],
)

print("Agent with snapshot-based interpreter created.")

## 8. 安全边界

| 能力 | 默认可用 | 如何启用 |
|------|:--------:|----------|
| JavaScript 执行 | ✅ | 添加解释器中间件 |
| Top-level await | ✅ | 在解释器中使用 Promise |
| console.log 捕获 | ✅ | 通过 `capture_console=False` 禁用 |
| Agent 工具（PTC） | ❌ | 添加 PTC 白名单 |
| 解释器技能模块 | ❌ | 添加模块条目并配置 `skills_backend` |
| 文件系统访问 | ❌ | 通过 PTC 白名单暴露内置文件系统工具 |
| 网络访问 | ❌ | 通过 PTC 暴露特定网络工具 |
| 时钟/时间访问 | ❌ | 如需要，暴露显式时间工具 |
| Shell 命令/OS 操作 | ❌ | 使用沙箱 Backend |

> 解释器代码在嵌入的 QuickJS 上下文中运行，**非独立 VM 或进程**。将其视为**能力范围受限的执行层**，而非宿主内存隔离边界。对于不受信任的代码，应将 Agent 运行在隔离的工作进程或容器中，并保持 PTC 白名单狭窄。

## 9. 中间件配置选项

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `memory_limit` | 64 MB | QuickJS 堆内存限制 |
| `timeout` | 5.0 | 每次 eval 的超时秒数 |
| `max_ptc_calls` | 256 | 每次 eval 最大 `tools.*` 调用次数 |
| `tool_name` | `"eval"` | 暴露给模型的解释器工具名 |
| `max_result_chars` | 4000 | 结果和 stdout 最大返回字符数 |
| `capture_console` | True | 是否捕获 `console.log/warn/error` |
| `ptc` | None | PTC 白名单：工具名列表或 BaseTool 实例 |
| `skills_backend` | None | 用于解析解释器技能模块的 Backend |
| `snapshot_between_turns` | True | 是否跨轮次保存解释器状态快照 |
| `max_snapshot_bytes` | None | 最大序列化快照大小，默认 = memory_limit |

---
**参考链接**

- [Interpreters — LangChain Docs](https://docs.langchain.com/oss/python/deepagents/interpreters)
- [Deep Agents Skills](https://docs.langchain.com/oss/python/deepagents/skills)
- [Sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes)